# model/01: CRN Encoder

This notebook explains the bipartite graph neural network that converts a CRN into a dense vector representation for downstream use by the neural SDE.

**Contents:**
1. Bipartite Graph Structure — EdgeFeature, edge extraction
2. Species and Reaction Embeddings — how initial embeddings are constructed
3. BipartiteGNNEncoder — full encoder forward pass, CRNContext shapes
4. Encoder Invariance — same CRN, same context vector regardless of downstream protocol
5. Encoding Different CRNs — context vectors differ across network topologies

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

from _shared.plotting import setup_style

setup_style()

import torch
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

## 1. Bipartite Graph Structure

A CRN defines a bipartite graph between two node types:
- **Species nodes** — molecular populations
- **Reaction nodes** — kinetic processes

Edges connect reactions to species they affect (via stoichiometry) or depend on (via propensity). Each edge carries feature channels that capture this relationship.

The `EdgeFeature` enum lists the channels stored per edge.

In [2]:
from crn_surrogate.data.generation.reference_crns import lotka_volterra
from crn_surrogate.encoder.tensor_repr import crn_to_tensor_repr
from crn_surrogate.encoder.graph_utils import EdgeFeature

crn_lv = lotka_volterra(k_prey_birth=1.0, k_predation=0.005, k_predator_death=0.6)
crn_lv_repr = crn_to_tensor_repr(crn_lv)

edges = crn_lv_repr.bipartite_edges

print(f"CRN: {crn_lv}")
print(f"Species: {crn_lv.species_names}")
print(f"\nEdge feature dim: {edges.edge_feat_dim}")
print(f"Feature channels: {[f.name for f in EdgeFeature]}")
print()
print("Reaction \u2194 Species edges:")
print(f"  edge index shape: {edges.rxn_to_species_index.shape}  (2, n_edges)")
for i in range(edges.rxn_to_species_index.shape[1]):
    r = edges.rxn_to_species_index[0, i].item()
    s = edges.rxn_to_species_index[1, i].item()
    feat = edges.rxn_to_species_feat[i]
    sname = crn_lv.species_names[s]
    net   = feat[EdgeFeature.NET_CHANGE].item()
    stoic = feat[EdgeFeature.IS_STOICHIOMETRIC].item()
    dep   = feat[EdgeFeature.IS_DEPENDENCY].item()
    print(f"  rxn {r} \u2194 {sname}: net_change={net:+.0f}  is_stoich={stoic:.0f}  is_dep={dep:.0f}")

CRN: CRN(n_species=2, n_reactions=3, species=('prey', 'predator'))
Species: ('prey', 'predator')

Edge feature dim: 3
Feature channels: ['NET_CHANGE', 'IS_STOICHIOMETRIC', 'IS_DEPENDENCY']

Reaction ↔ Species edges:
  edge index shape: torch.Size([2, 4])  (2, n_edges)
  rxn 0 ↔ prey: net_change=+1  is_stoich=1  is_dep=1
  rxn 1 ↔ prey: net_change=-1  is_stoich=1  is_dep=1
  rxn 1 ↔ predator: net_change=+1  is_stoich=1  is_dep=1
  rxn 2 ↔ predator: net_change=-1  is_stoich=1  is_dep=1


## 2. Species and Reaction Embeddings

Before message passing, each node is assigned an initial embedding:

- **Species nodes**: a small MLP that takes the initial molecule count and an `is_external` flag (1 for external/input species, 0 for internal)
- **Reaction nodes**: a lookup table over propensity type ID plus a linear projection of propensity parameters

The `is_external` feature is the boundary between the CRN's intrinsic structure and the experimental protocol.

In [3]:
from crn_surrogate.crn.crn import CRN
from crn_surrogate.crn.reaction import Reaction
from crn_surrogate.crn.propensities import hill, mass_action
from crn_surrogate.crn.inputs import InputProtocol, single_pulse

# CRN with one external species (inducer)
crn_ext = CRN(
    reactions=[
        Reaction(
            stoichiometry=torch.tensor([1, 0]),
            propensity=hill(v_max=5.0, k_m=10.0, hill_coefficient=2.0, species_index=1),
            name="A production",
        ),
        Reaction(
            stoichiometry=torch.tensor([-1, 0]),
            propensity=mass_action(0.2, torch.tensor([1.0, 0.0])),
            name="A degradation",
        ),
    ],
    species_names=["A", "I"],
    external_species=frozenset({1}),
)

crn_ext_repr = crn_to_tensor_repr(crn_ext)

print(f"CRN: {crn_ext}")
print(f"is_external tensor: {crn_ext_repr.is_external}")
print(f"propensity_type_ids: {crn_ext_repr.propensity_type_ids.tolist()}")
print(f"propensity_params shape: {crn_ext_repr.propensity_params.shape}  (n_reactions, max_params)")
print(f"propensity_params:\n{crn_ext_repr.propensity_params}")

CRN: CRN(n_species=2, n_reactions=2, species=('A', 'I'))
is_external tensor: tensor([False,  True])
propensity_type_ids: [1, 0]
propensity_params shape: torch.Size([2, 8])  (n_reactions, max_params)
propensity_params:
tensor([[ 5.0000, 10.0000,  2.0000,  1.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.2000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000]])


## 3. BipartiteGNNEncoder

The `BipartiteGNNEncoder` runs L rounds of bipartite message passing:
1. Reactions send messages to species (weighted by edge features)
2. Species send messages back to reactions

After L rounds, species and reaction embeddings are pooled into a global context vector. The output is a `CRNContext` containing per-species embeddings, per-reaction embeddings, and a pooled context vector.

In [4]:
from crn_surrogate.configs.model_config import EncoderConfig
from crn_surrogate.encoder.bipartite_gnn import BipartiteGNNEncoder

enc_config = EncoderConfig(d_model=32, n_layers=4)
encoder = BipartiteGNNEncoder(enc_config)

n_params = sum(p.numel() for p in encoder.parameters())
print(f"Encoder parameters: {n_params:,}")

init_state = torch.tensor([5.0, 0.0])  # A=5, I=0
ctx = encoder(crn_ext_repr, init_state)

print(f"\nspecies_embeddings : {ctx.species_embeddings.shape}  (n_species, d_model)")
print(f"reaction_embeddings: {ctx.reaction_embeddings.shape}  (n_reactions, d_model)")
print(f"context_vector     : {ctx.context_vector.shape}  (2 * d_model)")
print(f"context_vector norm: {ctx.context_vector.norm().item():.4f}")

Encoder parameters: 20,656

species_embeddings : torch.Size([2, 32])  (n_species, d_model)
reaction_embeddings: torch.Size([2, 32])  (n_reactions, d_model)
context_vector     : torch.Size([64])  (2 * d_model)
context_vector norm: 5.4791


## 4. Encoder Invariance

The encoder encodes CRN structure and initial state — it knows nothing about what protocol will be applied downstream. Encoding the same CRN twice must give the same output.

This is the key invariant that makes the design composable: one encoder pass produces a context vector that can be reused across all protocol variants of the same experiment.

In [5]:
encoder.eval()

with torch.no_grad():
    ctx_run1 = encoder(crn_ext_repr, init_state)
    ctx_run2 = encoder(crn_ext_repr, init_state)

max_diff = (ctx_run1.context_vector - ctx_run2.context_vector).abs().max().item()
print(f"Max difference between two encoder runs (same input): {max_diff:.2e}  (should be 0)")

# Different initial states produce different context vectors
initial_state_alt = torch.tensor([20.0, 0.0])
with torch.no_grad():
    ctx_alt = encoder(crn_ext_repr, initial_state_alt)

max_diff_vs_alt = (ctx_run1.context_vector - ctx_alt.context_vector).abs().max().item()
print(f"Max difference (A=5 vs A=20): {max_diff_vs_alt:.4f}  (expected nonzero)")

Max difference between two encoder runs (same input): 0.00e+00  (should be 0)
Max difference (A=5 vs A=20): 0.3069  (expected nonzero)


## 5. Encoding Different CRNs

Context vectors for structurally different CRNs should differ. Here we encode three CRNs (birth-death, toggle switch, and the external-input CRN from above) and compare cosine similarities between their context vectors.

In [6]:
from crn_surrogate.data.generation.reference_crns import birth_death, toggle_switch
import torch.nn.functional as F

crns_to_compare = [
    (birth_death(k_birth=2.0, k_death=0.5), torch.tensor([0.0]),     "Birth-Death"),
    (toggle_switch(),                        torch.tensor([5.0, 5.0]), "Toggle Switch"),
    (crn_ext,                                torch.tensor([5.0, 0.0]), "BD + External"),
]

reprs = [crn_to_tensor_repr(crn) for crn, _, _ in crns_to_compare]
labels_list = [label for _, _, label in crns_to_compare]
init_states = [state for _, state, _ in crns_to_compare]

encoder.eval()
with torch.no_grad():
    contexts = [encoder(r, s).context_vector for r, s in zip(reprs, init_states)]

print("Cosine similarity matrix:")
print(f"{'':20}", end="")
for label in labels_list:
    print(f"{label:>16}", end="")
print()
for i, (label_i, ctx_i) in enumerate(zip(labels_list, contexts)):
    print(f"{label_i:20}", end="")
    for ctx_j in contexts:
        sim = F.cosine_similarity(ctx_i.unsqueeze(0), ctx_j.unsqueeze(0)).item()
        print(f"{sim:>16.4f}", end="")
    print()

print("\nDiagonal = 1.0 (self-similarity). Off-diagonal < 1.0 (different CRNs).")

Cosine similarity matrix:
                         Birth-Death   Toggle Switch   BD + External
Birth-Death                   1.0000          0.5518          0.5296
Toggle Switch                 0.5518          1.0000          0.7668
BD + External                 0.5296          0.7668          1.0000

Diagonal = 1.0 (self-similarity). Off-diagonal < 1.0 (different CRNs).


## Summary

This notebook showed:

- The bipartite graph structure of a CRN: species nodes, reaction nodes, and edge features capturing stoichiometry and dependency
- Species embeddings that include the `is_external` flag, creating a clear boundary between network structure and experimental protocol
- The `BipartiteGNNEncoder` producing `CRNContext` with per-species, per-reaction, and global context vectors
- Deterministic encoding: the same CRN at the same initial state always maps to the same context vector
- Different CRN topologies produce distinct context vectors (different cosine similarities)